In [1]:
import os
import json 
import argparse 
import subprocess
import random
import pandas as pd
import requests
import zipfile
from typing import Dict, List, Any
from kaggle_secrets import UserSecretsClient

In [2]:
try: 
    from datasets import load_dataset 
except ImportError: 
    raise ImportError("Please install Hugging Face datasets: pip install datasets")

In [3]:
def random_sample(data, num_samples=100, seed=42):
    """Randomly sample items from a dataset."""
    rng = random.Random(seed)

    if num_samples > len(data):
        raise ValueError(
            f"Requested {num_samples} samples, "
            f"but only {len(data)} items are available."
        )

    return rng.sample(data, num_samples)

In [4]:
def extract_xquad(num_samples: int = 100) -> List[Dict[str, Any]]: 
    """ 
    Extracts parallel XQuAD items across Turkish (xquad.tr), English (xquad.en), and German (xquad.de). 
    """ 
    
    print("Loading XQuAD splits (xquad.tr, xquad.en, xquad.de)...") 
    ds_tr = load_dataset("xquad", "xquad.tr", split="validation") 
    ds_en = load_dataset("xquad", "xquad.en", split="validation") 
    ds_de = load_dataset("xquad", "xquad.de", split="validation") 
    
    extracted = [] 
    limit = min(num_samples, len(ds_tr), len(ds_en), len(ds_de)) 
    
    for i in range(limit): 
        item_tr = ds_tr[i] 
        item_en = ds_en[i] 
        item_de = ds_de[i] 
        
        extracted.append({ 
            "id": f"xquad_{i}", 
            "task": "SpanQA", 
            "turkish": { 
                "context": item_tr["context"], 
                "question": item_tr["question"], 
                "answers": item_tr["answers"]["text"] 
            }, 
            "english": { 
                "context": item_en["context"], 
                "question": item_en["question"], 
                "answers": item_en["answers"]["text"] 
            }, 
            "german": { 
                "context": item_de["context"], 
                "question": item_de["question"], 
                "answers": item_de["answers"]["text"] 
            } 
        }) 
        
    print(f"Extracted {len(extracted)} parallel XQuAD items.") 
    return extracted

In [5]:
def extract_xnli(num_samples: int = 100) -> List[Dict[str, Any]]: 
    """ 
    Extracts parallel XNLI items across Turkish (tr), English (en), and German (de). 
    Labels: 0 = entailment, 1 = neutral, 2 = contradiction. 
    """ 
    
    print("Loading XNLI splits (tr, en, de)...") 
    
    ds_tr = load_dataset("xnli", "tr", split="validation") 
    ds_en = load_dataset("xnli", "en", split="validation") 
    ds_de = load_dataset("xnli", "de", split="validation") 
    
    extracted = [] 
    limit = min(num_samples, len(ds_tr), len(ds_en), len(ds_de)) 
    
    for i in range(limit): 
        item_tr = ds_tr[i] 
        item_en = ds_en[i] 
        item_de = ds_de[i] 
        
        extracted.append({ 
            "id": f"xnli_{i}", 
            "task": "NLI", 
            "label": item_tr["label"], # 0,1, or 2
            "turkish": { 
                "premise": item_tr["premise"], 
                "hypothesis": item_tr["hypothesis"]
            }, 
            "english": { 
                "premise": item_en["premise"], 
                "hypothesis": item_en["hypothesis"] 
            }, 
            "german": { 
                "premise": item_de["premise"], 
                "hypothesis": item_de["hypothesis"] 
            } 
        }) 
        
    print(f"Extracted {len(extracted)} parallel XNLI items.") 
        
    return extracted

In [6]:
def extract_belebele(num_samples: int = 100) -> List[Dict[str, Any]]:
    """
    Extract parallel Belebele reading comprehension items (tur_Latn, eng_Latn, deu_Latn).
    """
   
    print("Loading Belebele splits (tur_Latn, eng_Latn, deu_Latn)...")

    ds_tr = load_dataset("facebook/belebele", "tur_Latn", split="test").to_pandas()
    ds_en = load_dataset("facebook/belebele", "eng_Latn", split="test").to_pandas()
    ds_de = load_dataset("facebook/belebele", "deu_Latn", split="test").to_pandas()

    ds_en=ds_en.add_prefix("en_")
    ds_de=ds_de.add_prefix("de_")

    ds_en= ds_en.rename(columns={"en_link": "link", "en_question_number":"question_number"})
    ds_de= ds_de.rename(columns={"de_link": "link", "de_question_number":"question_number"})

    merged_df = pd.merge(ds_tr, ds_en, on=["link", "question_number"], how="inner")
    merged_df = pd.merge(merged_df, ds_de, on=["link", "question_number"], how="inner")

    extracted: List[Dict[str, Any]] = []
    limit = min(num_samples,len(merged_df))

    for i,row in merged_df.head(limit).iterrows():
        extracted.append(
            {
                "id": f"belebele_{i}",
                "task": "ReadingComprehensionReasoning",
                "link": row["link"],
                "correct_answer_num": row["correct_answer_num"], # "1", "2", "3", or "4"
                "turkish": {
                    "passage": row["flores_passage"],
                    "question": row["question"],
                    "mc_answer1": row["mc_answer1"],
                    "mc_answer2": row["mc_answer2"],
                    "mc_answer3": row["mc_answer3"],
                    "mc_answer4": row["mc_answer4"],
                },
                "english": {
                    "passage": row["en_flores_passage"],
                    "question": row["en_question"],
                    "mc_answer1": row["en_mc_answer1"],
                    "mc_answer2": row["en_mc_answer2"],
                    "mc_answer3": row["en_mc_answer3"],
                    "mc_answer4": row["en_mc_answer4"],
                },
                "german": {
                    "passage": row["de_flores_passage"],
                    "question": row["de_question"],
                    "mc_answer2": row["de_mc_answer2"],
                    "mc_answer3": row["de_mc_answer3"],
                    "mc_answer4": row["de_mc_answer4"],
                    "mc_answer1": row["de_mc_answer1"],
                },
            }
        )

    print(f"Extracted {len(extracted)} parallel Belebele items.")
    return extracted

In [7]:
url = "https://storage.googleapis.com/gresearch/presto/presto_v1.zip"
zip_path = "/kaggle/working/presto_v1.zip"
extract_path = "/kaggle/working/presto"

In [8]:
def download_presto():
    if os.path.exists(extract_path):
        print("PRESTO already downloaded and extracted.")
        return
    
    print("Downloading PRESTO...")
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    
    with open(zip_path, "wb") as f:
        f.write(response.content)

    print("Extracting PRESTO...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_path)
    
    print("PRESTO downloaded and extracted.")

In [9]:
def get_presto_utterances(locale_path: str, limit: int) -> List[str]:
    """Reads a jsonl file and extracts user utterances."""
    
    utterances = []
    if not os.path.exists(locale_path):
        print(f"WARNING: File not found at {locale_path}")
        return utterances
        
    with open(locale_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                data = json.loads(line)

                text= data.get("inputs")

                if text:
                    utterances.append(text)
                
            except Exception as e:
                pass
                
            if len(utterances) >= limit:
                break
    return utterances

In [10]:
def extract_presto(num_samples: int = 100) -> List[Dict[str, Any]]:
    """
    Extract parallel Presto items (eng, deu) from local files.
    For the History (H) condition, only English and German are needed.
    """

    print("Loading local Presto files for en-US and de-DE (monolingual)...")

    # Paths to monolungial test sets
    en_path = os.path.join(extract_path, "test_partitions", "en-US", "en-US_no_phenomena", "test.jsonl")
    de_path = os.path.join(extract_path, "test_partitions", "de-DE", "de-DE_no_phenomena", "test.jsonl")

    # Path to the actual code-mixed dataset
    de_cm_path = os.path.join(extract_path, "test_partitions", "de-DE", "de-DE_code-mixing", "test.jsonl")
    
    en_utterances = get_presto_utterances(en_path, num_samples)
    de_utterances = get_presto_utterances(de_path, num_samples)
    cm_utterances = get_presto_utterances(de_cm_path, num_samples)
    
    extracted: List[Dict[str, Any]] = []
    limit = min(num_samples, len(en_utterances), len(de_utterances))

    if limit == 0:
        print("WARNING: Could not find enough PRESTO data. Check file paths.")
        return extracted
    
    for i in range(limit):
         extracted.append(
             {
                 "id": f"presto_{i}",
                 "task": "TaskOrientedDialogue",
                 "english": {
                     "query": en_utterances[i],
                     "response": "Action completed successfully."
                 },
                 "german": {
                     "query": de_utterances[i],
                     "response": "Aktion erfolgreich abgeschlossen."
                 },
                "german-code-mixed": {
                    "query": cm_utterances[i],
                    "response": "Aktion erfolgreich completed."
                }
             }
         )

    print(f"Extracted {len(extracted)} parallel Presto items.")
    return extracted

In [11]:
SAVE_FOLDER = "code_switching_test_data"
DATASET_NAME = "code-switching-test-data"
KAGGLE_USERNAME = "ozlemkrblt"
KAGGLE_SECRET_NAME = "my-secret"
HF_TOKEN = "hf-token"

In [12]:
TASK_EXTRACTORS = {
    "xnli": extract_xnli,
    "belebele": extract_belebele,
    "xquad": extract_xquad,
    "presto": extract_presto
}

In [13]:
def save_json(data: List[Dict[str, Any]], filepath: str):
    """Save data as JSON."""
    os.makedirs(os.path.dirname(filepath), exist_ok=True)

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(data)} items to '{filepath}'.")

In [14]:
def setup_kaggle_auth():
    """Set up Kaggle API authentication."""
    user_secrets = UserSecretsClient()
    os.environ["KAGGLE_KEY"] = user_secrets.get_secret(KAGGLE_SECRET_NAME)
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["HF_TOKEN"] = HF_TOKEN

In [15]:
def create_or_update_kaggle_dataset(save_folder: str):
    """Create or update a Kaggle dataset from the saved directory."""

    dataset_ref = f"{KAGGLE_USERNAME}/{DATASET_NAME}"
    metadata_path = os.path.join(save_folder, "dataset-metadata.json")

    # Initialize metadata if it does not exist
    if not os.path.exists(metadata_path):
        subprocess.run(
            ["kaggle", "datasets", "init", "-p", save_folder],
            check=True
        )

    # Read metadata
    with open(metadata_path, "r", encoding="utf-8") as f:
        dataset_meta = json.load(f)

    # Update metadata
    dataset_meta["id"] = dataset_ref
    dataset_meta["title"] = DATASET_NAME

    # Format the task list as a string and assign it to the subtitle or description
    task_list_string = ", ".join(TASK_EXTRACTORS.keys())
    dataset_meta["description"] = f"Test data suites for tasks: {task_list_string}"

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(
            dataset_meta,
            f,
            ensure_ascii=False,
            indent=2
        )
    
    # Check whether the dataset already exists
    result = subprocess.run(
        ["kaggle", "datasets", "files", dataset_ref],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print(f"\nDataset '{dataset_ref}' already exists.")
        print("Updating dataset...")
    
        subprocess.run(
            [
                "kaggle",
                "datasets",
                "version",
                "-p",
                save_folder,
                "-m",
                "Updated test data"
            ],
            check=True
        )

        print("Kaggle dataset updated successfully.")

    else:
        print(f"\nDataset '{dataset_ref}' does not exist.")
        print("Creating dataset...")
        
        # Create Kaggle dataset
        subprocess.run(
            [
                "kaggle",
                "datasets",
                "create",
                "-p",
                save_folder],
            check=True
        )

    print(
        f"\nKaggle dataset created successfully:\n"
        f"/kaggle/input/datasets/{KAGGLE_USERNAME}/{DATASET_NAME}"
    )


In [16]:
def main():     
    parser = argparse.ArgumentParser(description= "Extract parallel TR-EN-DE benchmark items into task-specific JSON files.")
    
    parser.add_argument(
        "--num_samples",
        type=int,
        default=100,
        help="Number of items per task") 
    
    parser.add_argument(
        "--output_dir",
        type=str,
        default=SAVE_FOLDER,
        help="Output directory") 
    
    args = parser.parse_args(
        args=['--num_samples', '100',
              '--output_dir',SAVE_FOLDER])

    # Make sure directory exists
    os.makedirs(args.output_dir,exist_ok=True)

    setup_kaggle_auth()
    download_presto()

    # Save every dataset as a JSON file
    for task_name, extract_function in TASK_EXTRACTORS.items():

        print(f"\nExtracting {task_name}...")

        data = extract_function(args.num_samples)
        
        # Randomly select 100 items
        data = random_sample(
            data,
            num_samples=args.num_samples,
            seed=42
        )

        filepath = os.path.join(
            args.output_dir,
            f"{task_name}_suite.json"
        )

        save_json(data, filepath)

    print(
        f"\nAll task suites generated inside "
        f"'{args.output_dir}/'!"
    )

    # Upload the whole directory to Kaggle
    create_or_update_kaggle_dataset(args.output_dir)


if __name__ == "__main__": 
        main()

Extracting PRESTO...
PRESTO downloaded and extracted.

Extracting xnli...
Loading XNLI splits (tr, en, de)...


README.md: 0.00B [00:00, ?B/s]

tr/train-00000-of-00001.parquet:   0%|          | 0.00/48.0M [00:00<?, ?B/s]

tr/test-00000-of-00001.parquet:   0%|          | 0.00/338k [00:00<?, ?B/s]

tr/validation-00000-of-00001.parquet:   0%|          | 0.00/172k [00:00<?, ?B/s]

...

Upload successful: xnli_suite.json (64KB)
Starting upload for file presto_suite.json


100%|██████████| 47.8k/47.8k [00:00<00:00, 225kB/s]
  0%|          | 0.00/259k [00:00<?, ?B/s]

Upload successful: presto_suite.json (48KB)
Starting upload for file xquad_suite.json


100%|██████████| 259k/259k [00:00<00:00, 1.39MB/s]


Upload successful: xquad_suite.json (259KB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/ozlemkrblt/code-switching-test-data
Kaggle dataset updated successfully.

Kaggle dataset created successfully:
/kaggle/input/datasets/ozlemkrblt/code-switching-test-data


In [17]:
def erase_folder():
    '''Code to erase unnecessary output folders'''
    import shutil

    shutil.rmtree(SAVE_FOLDER, ignore_errors=True)